# Best-member retrieval against the verified catalogue

**The question.** Of the crops that never entered the catalogue, which are impressions of a
catalogue design it already holds?

**Why it can be asked at all.** Chapter 2 §9 counted what its clustering discarded: 2,455 crops
the selected method rejected are verified catalogue members, reached instead by a second grouping method on
identical vectors. The remainder is therefore known to be productive rather than hoped to be.

**Why the operating point can move.** Clustering had no labels, so it had to infer both how many
identities exist and which crops belong to each, and the safe policy under that ignorance is to
group where density is unambiguous and refuse otherwise. Here each class is a verified exemplar
set, so the question has a reference to check against.

**What it produces.** A score for every crop outside the catalogue, ranked review folders, and,
after review, the confirmed recoveries and their reviewer-confirmation rate.

**How to read it.** Sections 2 to 4 build the pool and the scoring rule. Sections 5 and 6 size the
review and export it. Section 7 states how review was carried out and what was wrong with it.
Section 8 reports the outcome, and Section 9 records a frozen follow-up after catalogue revision.

Retrieval proposes; the reviewer disposes. Nothing enters the catalogue without confirmation.

---

**Reads** the verified catalogue and the 14,745 uncatalogued crops · **Writes**
`3_retrieval_outputs/` · **Chapter README** §3, §4

## 1. Configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
assert NOTEBOOK_DIR.name == '3_retrieval', 'Run this notebook from its own folder, 3_retrieval/.'
PROJECT_DIR = NOTEBOOK_DIR.parent

FEATURE_RUN   = 'all_regions_v1_minside24_dinov2_vitb14_binarized_v1'
FEATURE_DIR   = PROJECT_DIR / 'feature_extraction_outputs' / FEATURE_RUN
FEATURES_PATH = FEATURE_DIR / 'dino_features_binarized.npy'
MANIFEST_PATH = FEATURE_DIR / 'features_manifest.csv'

CATALOGUE_DIR = PROJECT_DIR / 'Fleurons' / 'Fleurons'      # 89 verified classes

RUN_TAG    = 'v1'
OUTPUT_DIR = PROJECT_DIR / '3_retrieval_outputs' / f'centroid_match_{RUN_TAG}'
REVIEW_DIR = OUTPUT_DIR / 'review'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TOP_K     = 40     # candidates exported per class for review
MIN_SCORE = 0.80   # below this a match is not worth a reviewer's time

print('catalogue :', CATALOGUE_DIR)
print('writing to:', OUTPUT_DIR.relative_to(PROJECT_DIR))

catalogue : <project>/Fleurons/Fleurons
writing to: 3_retrieval_outputs/centroid_match_v1


## 2. Load the catalogue and the feature space

Each class folder holds symlinks to crop files. The crop **basename** is the join key
between the catalogue and the feature manifest.

The stored descriptors are not unit-length, so they are L2-normalised here; cosine
similarity is then a dot product.

In [2]:
import os

catalogue = {}
for folder in sorted(CATALOGUE_DIR.iterdir()):
    if not folder.is_dir():
        continue
    for f in folder.iterdir():
        name = Path(os.readlink(f)).name if f.is_symlink() else f.name
        catalogue[name] = folder.name

manifest = pd.read_csv(MANIFEST_PATH)
manifest['crop_name'] = manifest.crop_path.map(lambda p: Path(p).name)
assert not manifest.crop_name.duplicated().any(), 'Crop basenames must be unique join keys.'

features = np.load(FEATURES_PATH).astype(np.float32)
assert len(manifest) == len(features)
assert np.array_equal(manifest.feature_index.to_numpy(), np.arange(len(manifest)))
norms = np.linalg.norm(features, axis=1, keepdims=True)
assert (norms > 0).all(), 'Zero-norm descriptors cannot be cosine-normalised.'
features /= norms

manifest['klass'] = manifest.crop_name.map(catalogue)

print(f'catalogue      : {len(set(catalogue.values()))} classes, {len(catalogue)} crops')
print(f'feature matrix : {features.shape}')
print(f'catalogue crops located in the matrix: {manifest.klass.notna().sum()}')

catalogue      : 89 classes, 7013 crops
feature matrix : (21750, 768)
catalogue crops located in the matrix: 7005


## 3. Candidate pool

Every crop in the feature matrix that is **not** already in the catalogue: **14,745** of the
21,750 embedded crops.

This is not the same as the material clustering left unassigned, and the difference matters. The
catalogue is a hand-curated artefact drawn from two grouping passes, so some crops the clustering
assigned are outside it and some it rejected are inside it. Chapter 2 §9 gives the four-way split;
the pool searched here is its bottom row:

| | count | what it is |
|---|---:|---|
| never grouped | 12,135 | the clustering rejected it and no reviewer picked it up |
| grouped, then dropped | 2,610 | the clustering placed it in a cluster and a reviewer removed it |

The second group makes this a selected, difficult pool. A proposal drawn from it asks the reviewer
to overturn a judgement they have already made once; this does not make the observed shortlist
precision a formal lower bound on another population.

In [3]:
is_cat    = manifest.klass.notna().to_numpy()
cand_idx  = np.flatnonzero(~is_cat)
cand_feat = features[cand_idx]

print(f'catalogue crops : {is_cat.sum():>6}')
print(f'candidate pool  : {len(cand_idx):>6}')

catalogue crops :   7005
candidate pool  :  14745


## 4. Scoring: best match to any class member

For a candidate *c* and a class *K*, the score is

$$s(c, K) = \max_{m \in K} \; \cos(c, m)$$

**Not the distance to a class centroid.** The catalogue-correctness pass reported in Chapter 2 README §8 measured that
centroid matching buries true matches, because a large class with mixed contrast has a
smeared mean that sits near nothing, one confirmed match ranked 9th by centroid and 1st
by best-member. Best-member also behaves correctly for classes that legitimately contain
visual variation from inking and wear.

Each candidate keeps its single best class, the score, and which member it matched, so
every proposal can be traced to the exemplar that justified it.

In [4]:
classes    = sorted(set(catalogue.values()))
best_score = np.full(len(cand_idx), -1.0, dtype=np.float32)
best_class = np.empty(len(cand_idx), dtype=object)
best_match = np.empty(len(cand_idx), dtype=object)

for klass in classes:
    rows = manifest.index[manifest.klass == klass].to_numpy()
    if len(rows) == 0:
        continue
    sims = cand_feat @ features[rows].T          # (candidates, members)
    arg  = sims.argmax(axis=1)
    top  = sims[np.arange(len(sims)), arg]
    hit  = top > best_score
    best_score[hit] = top[hit]
    best_class[hit] = klass
    best_match[hit] = manifest.crop_name.to_numpy()[rows][arg[hit]]

results = pd.DataFrame({
    'feature_index': manifest.feature_index.to_numpy()[cand_idx],
    'crop_name'    : manifest.crop_name.to_numpy()[cand_idx],
    'crop_path'    : manifest.crop_path.to_numpy()[cand_idx],
    'best_class'   : best_class,
    'score'        : best_score,
    'matched_crop' : best_match,
}).sort_values('score', ascending=False)

results.to_csv(OUTPUT_DIR / 'retrieval_scores.csv', index=False)
print(f'scored {len(results)} candidates')
print(results.score.describe().round(3).to_string())

scored 14745 candidates
count    14745.000
mean         0.842
std          0.078
min          0.301
25%          0.810
50%          0.858
75%          0.896
max          0.974


## 5. How many candidates each threshold would surface

Retrieval ranks; it does not decide. This table shows the review burden implied by each
operating point, before any threshold is chosen. The planned calibration/held-out threshold
selection was not executed; the run instead exported the bounded top-40-per-class shortlist
above 0.80, as documented in the chapter README §3.

In [5]:
bands = [0.98, 0.95, 0.92, 0.90, 0.88, 0.85, 0.80]
rows  = [{'threshold': t,
          'candidates': int((results.score >= t).sum()),
          'classes_hit': int(results.loc[results.score >= t, 'best_class'].nunique())}
         for t in bands]
print(pd.DataFrame(rows).to_string(index=False))

 threshold  candidates  classes_hit
      0.98           0            0
      0.95         178           39
      0.92        1487           65
      0.90        3267           73
      0.88        5247           76
      0.85        8130           79
      0.80       11541           82


## 6. Export ranked review folders

One folder per class, mirroring the curation layout already in use: symlinks named so the
**strongest matches sort to the top**. Reviewing means deleting what does not belong,
whatever survives is a confirmed recovery.

Only the top `TOP_K` candidates above `MIN_SCORE` are exported, so the review stays bounded.

In [6]:
# Guard: the review folders carry human decisions (deleted symlinks). Re-exporting
# would silently discard them, so this refuses to run over a reviewed directory.
ALREADY_REVIEWED = REVIEW_DIR.exists() and (OUTPUT_DIR / 'review_shortlist.csv').exists()

shortlist = results[results.score >= MIN_SCORE]
exported_rows = []
exported  = 0

for klass, grp in ([] if ALREADY_REVIEWED else shortlist.groupby('best_class')):
    grp = grp.nlargest(TOP_K, 'score')
    exported_rows.append(grp)
    dest = REVIEW_DIR / f'{klass}__{len(grp)}cand'
    dest.mkdir()
    for rank, row in enumerate(grp.itertuples(), start=1):
        src  = PROJECT_DIR / row.crop_path
        name = f'{rank:03d}_sim{round(row.score * 1000):04d}_{Path(row.crop_path).name}'
        if src.exists():
            os.symlink(src, dest / name)
            exported += 1

if ALREADY_REVIEWED:
    print('review directory already exists and holds human decisions - export skipped.')
    print('Raise RUN_TAG to export a fresh candidate set.')
else:
    # Only what was actually exported can be reviewed, so that is what gets scored.
    REVIEW_DIR.mkdir(parents=True, exist_ok=True)
    pd.concat(exported_rows).to_csv(OUTPUT_DIR / 'review_shortlist.csv', index=False)
    print(f'{exported} candidates exported across {len(list(REVIEW_DIR.iterdir()))} folders')

review directory already exists and holds human decisions - export skipped.
Raise RUN_TAG to export a fresh candidate set.


## 7. Review protocol, and deviations from it

Each folder is named for the catalogue class whose matches it proposes, and the files within it
are sorted best-first. Review is carried out in a file manager: a candidate that does not belong
to the class is deleted, and a candidate that survives is a confirmed recovery. The labels are
therefore carried by the contents of the folders rather than by a separate annotation file, which
is why the export above refuses to run over a directory that has already been reviewed. This is a
binary kept/deleted decision; the prospectively specified `same_fleuron`, `different`, and
`non_fleuron` labels were not recorded separately.

Two failure modes of the representation are known from earlier stages and bear on judgements at
scores near 0.90: the embedding cannot resolve lobe count on small florets, and it is
orientation-sensitive, so a rotated impression of the same fleuron can score *lower* than a
genuinely different fleuron of similar outline.

**Score cueing.** The prospectively specified protocol called for blind review. The exported
filenames encode each candidate's rank and score, so the reviewer saw both while deciding. Clear
matches are less vulnerable to this cue, but for ambiguous crops the score was visible, and
any tendency to resolve doubt in its favour would produce exactly the monotone gradient Section 8
reports. That correlation is therefore partly self-fulfilling and is not read as independent
evidence that the ranking works and borderline keep/delete decisions may also be affected.
`2_HoldOutValidation.ipynb` removes reviewer cueing but remains an internal closed-world check.
Stripping the score from filenames and shuffling within folders would remove this cue in a rerun.

## 8. What the review returned

Run after review. The surviving symlinks are read back, matched to the exported shortlist, and
scored by band. This is the chapter's headline operational result. Because the planned threshold
calibration/held-out split was not executed, it is not a full test of that protocol.

In [7]:
# Run after review. Reads back what survived and reports precision by score band.
survivors = set()
if REVIEW_DIR.exists():
    for folder in REVIEW_DIR.iterdir():
        if folder.is_dir():
            for f in folder.iterdir():
                survivors.add(Path(os.readlink(f)).name if f.is_symlink() else f.name)

sl = pd.read_csv(OUTPUT_DIR / 'review_shortlist.csv')
sl['confirmed'] = sl.crop_name.isin(survivors)
sl['band'] = pd.cut(sl.score, [0.80, 0.85, 0.88, 0.90, 0.92, 0.95, 1.01])

summary = (sl.groupby('band', observed=True)
             .agg(proposed=('confirmed', 'size'), confirmed=('confirmed', 'sum'))
             .assign(precision=lambda d: (d.confirmed / d.proposed).round(3)))
print(summary.to_string())
print(f'\ntotal confirmed recoveries: {int(sl.confirmed.sum())}')

              proposed  confirmed  precision
band                                        
(0.8, 0.85]        280         22      0.079
(0.85, 0.88]       252         30      0.119
(0.88, 0.9]        273         43      0.158
(0.9, 0.92]        552        138      0.250
(0.92, 0.95]       700        292      0.417
(0.95, 1.01]       177        139      0.785

total confirmed recoveries: 664


**Takeaway.** Of **2,234** candidates proposed across 82 classes, **664 were confirmed**, an
overall precision of **0.297**. Precision rises with score, from 0.079 in the lowest band to 0.785
in the highest, though for the reason given in Section 7 that gradient cannot be read as
independent validation of the ranking.

**The prospectively specified criterion is not demonstrated.** The chapter declared the pass worth reporting at
precision 0.90 or better at its operating point. Cumulatively, precision reaches 0.398 at a
threshold of 0.90 and 0.785 at 0.95; it reaches the bar only at 0.965, where twelve candidates
survive and the interval on that estimate runs from 0.646 to 0.985. Twelve candidates cannot
establish that the bar was met. **As an automatic thresholded classifier, retrieval fails the
criterion** on these in-sample cumulative summaries, and that is reported rather than repaired by moving the bar.

What the run directly supports is an operational yield: 664 recoveries from 2,234 proposals,
**one recovery per 3.4 proposals reviewed**, after inspecting 15.2% of the residual pool. Dividing
all 14,745 crops by the same 664 recoveries gives 22.2 only under a fixed-yield counterfactual; it
is not an observed random-review baseline.

## 9. Frozen follow-up after catalogue revision

After the main pass, three identities were added (`Fleuron_69`, `Fleuron_78`, `Fleuron_79`), two
labels were materially redefined (`Fleuron_35`, `Fleuron_9`), and `Fleuron_63` received a
class-specific follow-up because the global-best main rule had surfaced no proposal for it above
0.80. The candidate-generation code is no longer present. This section therefore reads the frozen
proposal rows and reconstructs their reviewed outcome; it does not reproduce candidate generation.

In [8]:
# Self-contained: this section reads a run that was carried out separately, so it resolves its
# own paths rather than inheriting them from the sections above.
RETRIEVAL_DIR = PROJECT_DIR / '3_retrieval_outputs'
GAP_DIR = RETRIEVAL_DIR / 'best_member_match_v1_gap_classes'
# The reviewed outcome is written where 3_OccurrenceTables.ipynb reads it. The directory keeps its
# original name from when this analysis lived in the hold-out notebook.
GAP_REVIEW_PATH = RETRIEVAL_DIR / 'holdout_v1' / 'gap_class_review.csv'
gap_candidates = pd.read_csv(GAP_DIR / 'candidates.csv')
gap_candidates['crop_name'] = gap_candidates.crop_path.map(lambda p: Path(p).name)

survivors = set()
for folder in sorted(p for p in (GAP_DIR / 'review').iterdir() if p.is_dir()):
    for link in folder.iterdir():
        survivors.add(Path(link.resolve()).name)

gap_candidates['confirmed'] = gap_candidates.crop_name.isin(survivors)
GAP_REVIEW_PATH.parent.mkdir(parents=True, exist_ok=True)
gap_candidates.to_csv(GAP_REVIEW_PATH, index=False)

outside = len(survivors - set(gap_candidates.crop_name))
print(f'classes searched          : {gap_candidates["class"].nunique()}')
print(f'proposal rows             : {len(gap_candidates)}')
print(f'unique candidate crops    : {gap_candidates.crop_name.nunique()}')
print(f'confirmed after review    : {int(gap_candidates.confirmed.sum())}')
print(f'precision of this pass    : {gap_candidates.confirmed.mean():.3f}')
print(f'links present in the review folders but never proposed by this run: {outside}')

classes searched          : 6
proposal rows             : 147
unique candidate crops    : 146
confirmed after review    : 32
precision of this pass    : 0.218
links present in the review folders but never proposed by this run: 2


**Takeaway.** The follow-up recovered **32 further unique impressions from 147 proposal rows
(146 unique crops)**, a row-level confirmation rate of 0.218, bringing confirmed recoveries across
both passes to **696**. It is not directly comparable to the main run: the frozen follow-up is
class-specific (one crop appears under two targets), whereas the main run assigns each crop to one
global best class. No causal class-size explanation is claimed without an exemplar-count control.

Two links sit in the review folders without appearing among this run's proposals, and the same
pattern appears at larger scale in the main run's review directory, which holds 39 such links. They
were placed there during curation from other sources. Every figure in this chapter counts only
candidates a run proposed and a reviewer then confirmed, so a naive count of surviving symlinks
would overstate the yield by exactly those links.